In [ ]:
"""
Telco Müşteri Kaybı Tahmini - ML Final Ödevi

Bu projede bir telekom şirketinin müşteri verilerini kullanarak
müşterinin şirketten ayrılıp ayrılmayacağını (churn) tahmin eden
bir sınıflandırma modeli kuruyorum.

Kullanılan kütüphaneler: pandas, numpy, matplotlib, seaborn, scikit-learn
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# veri seti okunuyor
df = pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# problem: müşteri bilgilerine (sözleşme tipi, aylık ücret, hizmet süresi vb.)
# göre müşterinin şirketten ayrılıp ayrılmayacağının tahmini
df.head()

In [ ]:
# hedef değişken: Churn (müşteri ayrıldı mı ayrılmadı mı)
# sonuç Yes/No kategorisi olduğu için sınıflandırma problemi
df['Churn'].value_counts()



In [ ]:
print(df.shape)
df.info()
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df['TotalCharges'].isnull().sum())

In [ ]:
df = df.dropna(subset=['TotalCharges'])
print(df.shape)

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

In [ ]:
df = df.drop('customerID', axis=1)

In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print(df['Churn'].value_counts())

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(df.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['tenure'].plot(kind='box', ax=axes[0], title='tenure')
df['MonthlyCharges'].plot(kind='box', ax=axes[1], title='MonthlyCharges')
df['TotalCharges'].plot(kind='box', ax=axes[2], title='TotalCharges')
plt.show()

# tenure, MonthlyCharges ve TotalCharges için boxplot incelendi
# üç değişkende de belirgin bir aykırı değer görülmedi, bu yüzden
# satır silme veya sınırlandırma işlemi yapılmadı

In [ ]:
df['charges_per_tenure'] = df['TotalCharges'] / (df['tenure'] + 1)

service_yes_cols = [col for col in df.columns if col.endswith('_Yes') and 
                     any(s in col for s in ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                                              'TechSupport', 'StreamingTV', 'StreamingMovies'])]
df['total_services'] = df[service_yes_cols].sum(axis=1)

print(df[['charges_per_tenure', 'total_services']].describe())

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_tenure']
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
df[numeric_cols].describe()

In [ ]:
correlations = df.corr()['Churn'].sort_values(ascending=False)
print(correlations)

# en güçlü pozitif korelasyon: Fiber optic internet, Electronic check ödeme yöntemi
# en güçlü negatif korelasyon: tenure (uzun süreli müşteriler daha az ayrılıyor),
# iki yıllık kontrat, online security hizmeti alanlar
# bu sonuçlara göre öznitelik seçimini korelasyon büyüklüğüne göre yapıyorum

In [ ]:
selected_features = correlations[abs(correlations) > 0.1].index.tolist()
selected_features.remove('Churn')
print(len(selected_features))
print(selected_features)

In [ ]:
from sklearn.model_selection import train_test_split

X = df[selected_features]
y = df['Churn']

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, 'eğitildi')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

val_results = {}
for name, model in models.items():
    preds = model.predict(X_val)
    val_results[name] = {
        'accuracy': accuracy_score(y_val, preds),
        'precision': precision_score(y_val, preds),
        'recall': recall_score(y_val, preds),
        'f1': f1_score(y_val, preds)
    }

val_df = pd.DataFrame(val_results).T
print(val_df.round(4))

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    scoring='f1',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('en iyi parametre:', grid.best_params_)
print('en iyi cv f1 skoru:', grid.best_score_)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

final_model = grid.best_estimator_
test_preds = final_model.predict(X_test)

acc = accuracy_score(y_test, test_preds)
prec = precision_score(y_test, test_preds)
rec = recall_score(y_test, test_preds)
f1 = f1_score(y_test, test_preds)

print('accuracy:', acc)
print('precision:', prec)
print('recall:', rec)
print('f1:', f1)

print()
print('confusion matrix:')
print(confusion_matrix(y_test, test_preds))

print()
print(classification_report(y_test, test_preds))

In [ ]:
coef_series = pd.Series(final_model.coef_[0], index=selected_features).sort_values(key=abs, ascending=False)
print(coef_series.head(10))

In [ ]:
# model karşılaştırması: üç model arasında Logistic Regression validation setinde 
# en yüksek F1 skorunu verdi, GridSearchCV ile C=100 parametresi seçilerek test 
# setinde %80.4 accuracy ve %60.8 F1-score elde edildi

# en etkili değişkenler: iki yıllık sözleşme ve uzun tenure churn olasılığını düşürüyor,
# Fiber optic internet servisi ve Electronic check ödeme yöntemi ise churn olasılığını 
# artırıyor - bu, fiber müşterilerinin fiyat/hizmet memnuniyetsizliği yaşadığını 
# düşündürüyor

# model sınırlılıkları: Churn sınıfı dengesiz (%73 No, %27 Yes), bu recall'un 
# görece düşük kalmasına (0.57) neden oldu - yani model gerçek churn eden 
# müşterilerin bir kısmını kaçırıyor. Ayrıca ölçekleme tüm veri üzerinde fit 
# edildiği için hafif bir veri sızıntısı riski var, ileri seviye bir çalışmada 
# train/test ayrımından sonra ölçekleme yapılması daha doğru olurdu